<p>
<img src="./imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">

# **Tutorial de AdaBoost: fundamentos, matemáticas e implementación con el dataset del golf**

## **1. Introducción**

**AdaBoost (Adaptive Boosting)** es un algoritmo de aprendizaje supervisado que permite construir un clasificador potente mediante la combinación de varios clasificadores sencillos, denominados *clasificadores débiles* (*weak learners*).

Fue propuesto por Yoav Freund y Robert Schapire en 1995 y constituye uno de los algoritmos fundamentales de los métodos de *boosting*.

Su principio de funcionamiento es:

> Entrenar clasificadores secuencialmente, modificando la importancia de los ejemplos de entrenamiento para que cada nuevo clasificador preste mayor atención a aquellos que resultan difíciles de clasificar.

AdaBoost tiene tres características fundamentales:

1. Entrena clasificadores débiles de forma secuencial.
2. Modifica los pesos de los ejemplos después de cada iteración.
3. Combina los clasificadores mediante una votación ponderada.

A lo largo de este tutorial utilizaremos el dataset clásico **Play Golf**, que permite predecir si se jugará al golf en función de las condiciones meteorológicas.

---

## **2. El dataset del golf**

El dataset contiene 14 observaciones, correspondientes a días con diferentes condiciones meteorológicas.

Cada observación tiene cuatro atributos:

- **Cielo:** soleado, nublado o lluvia.
- **Temperatura:** alta, suave o baja.
- **Humedad:** alta o normal.
- **Viento:** débil o fuerte.

La variable objetivo indica si se juega al golf.

| Día | Cielo | Temperatura | Humedad | Viento | Jugar |
|---|---|---|---|---|---|
| D1 | Soleado | Alta | Alta | Débil | No |
| D2 | Soleado | Alta | Alta | Fuerte | No |
| D3 | Nublado | Alta | Alta | Débil | Sí |
| D4 | Lluvia | Suave | Alta | Débil | Sí |
| D5 | Lluvia | Baja | Normal | Débil | Sí |
| D6 | Lluvia | Baja | Normal | Fuerte | No |
| D7 | Nublado | Baja | Normal | Fuerte | Sí |
| D8 | Soleado | Suave | Alta | Débil | No |
| D9 | Soleado | Baja | Normal | Débil | Sí |
| D10 | Lluvia | Suave | Normal | Débil | Sí |
| D11 | Soleado | Suave | Normal | Fuerte | Sí |
| D12 | Nublado | Suave | Alta | Fuerte | Sí |
| D13 | Nublado | Alta | Normal | Débil | Sí |
| D14 | Lluvia | Suave | Alta | Fuerte | No |

Tenemos nueve ejemplos positivos y cinco negativos.

Nuestro objetivo es aprender una función:

$$
H(x)\in\{-1,+1\}
$$

donde:

- $+1$: jugar al golf.
- $-1$: no jugar.

## **3. Clasificadores débiles**

Un clasificador débil es un modelo muy sencillo que, por sí solo, no tiene por qué ofrecer una gran precisión.

En este tutorial utilizaremos reglas basadas en una única condición meteorológica.

Por ejemplo:

**Clasificador 1**

- Si la humedad es normal, predecir «Sí».
- En caso contrario, predecir «No».

**Clasificador 2**

- Si el cielo está nublado, predecir «Sí».
- En caso contrario, predecir «No».

**Clasificador 3**

- Si el viento es débil, predecir «Sí».
- En caso contrario, predecir «No».

Estos clasificadores son deliberadamente simples.

La idea de AdaBoost no consiste en encontrar una única regla perfecta, sino en **combinar reglas que se complementen entre sí**.

---

## **4. Primera iteración de AdaBoost**

### **4.1. Inicialización de los pesos**

AdaBoost asigna un peso a cada ejemplo de entrenamiento.

Como tenemos $N=14$ observaciones, inicialmente todos reciben el mismo peso:

$$
w_i^{(1)}=\frac{1}{N}=\frac{1}{14}
$$

Por tanto:

$$
w_i^{(1)}\approx0.07143
$$

Los pesos representan la importancia relativa de cada ejemplo durante el entrenamiento.

En todo momento se normalizan para cumplir:

$$
\sum_{i=1}^{N}w_i=1
$$

### **4.2. Selección del primer clasificador**

AdaBoost evalúa diferentes clasificadores débiles y selecciona aquel que minimiza el error ponderado.

El error de un clasificador $h$ se define como:

$$
\epsilon=\sum_{i=1}^{N}w_i\,\mathbb{1}[h(x_i)\neq y_i]
$$

La función indicadora vale 1 cuando el clasificador se equivoca y 0 cuando acierta.

Para ilustrar el procedimiento, consideremos la regla:

> Si la humedad es normal, jugar. Si la humedad es alta, no jugar.

Esta regla falla en cuatro días:

- D3: predice No, pero se juega.
- D4: predice No, pero se juega.
- D6: predice Sí, pero no se juega.
- D12: predice No, pero se juega.

Como inicialmente todos los ejemplos tienen el mismo peso:

$$
\epsilon_1=4\cdot\frac1{14}
$$

$$
\boxed{\epsilon_1=\frac27\approx0.2857}
$$

Este será nuestro primer clasificador para la explicación.

**Nota:** esta regla se ha escogido para facilitar el ejemplo numérico. No estamos afirmando que sea necesariamente la primera regla seleccionada por una implementación que explore todas las reglas posibles.

### **4.3. Importancia del clasificador**

AdaBoost asigna a cada clasificador un coeficiente $\alpha_t$, que determina su influencia en la decisión final.

La fórmula es:

$$
\boxed{
\alpha_t=\frac12\ln\left(\frac{1-\epsilon_t}{\epsilon_t}\right)
}
$$

En nuestro caso:

$$
\alpha_1=\frac12\ln\left(\frac{1-2/7}{2/7}\right)
$$

$$
\alpha_1=\frac12\ln(2.5)
$$

$$
\boxed{\alpha_1\approx0.4581}
$$

Cuanto menor sea el error ponderado de un clasificador, mayor será su coeficiente.

Cuando el error se aproxima a 0,5, su coeficiente se aproxima a cero.

---

## **5. Actualización de los pesos**

Esta es una de las operaciones fundamentales de AdaBoost.

Una vez entrenado el clasificador, queremos que los ejemplos que ha clasificado incorrectamente tengan mayor importancia relativa en la siguiente iteración.

La actualización se realiza mediante:

$$
\boxed{
w_i^{(t+1)}
=
\frac{
w_i^{(t)}e^{-\alpha_t y_i h_t(x_i)}
}{
Z_t
}
}
$$

Donde:

- $w_i^{(t)}$: peso actual del ejemplo.
- $\alpha_t$: importancia del clasificador.
- $y_i$: etiqueta real, $+1$ o $-1$.
- $h_t(x_i)$: predicción del clasificador.
- $Z_t$: constante de normalización.

### **5.1. Cuando el clasificador acierta**

Si la predicción coincide con la etiqueta:

$$
y_i h_t(x_i)=+1
$$

Por tanto, antes de normalizar:

$$
w_i'=w_i e^{-\alpha_t}
$$

Como $\alpha_t>0$, el factor exponencial es menor que 1.

### **5.2. Cuando el clasificador falla**

Si la predicción es incorrecta:

$$
y_i h_t(x_i)=-1
$$

Entonces:

$$
w_i'=w_i e^{+\alpha_t}
$$

El factor exponencial es mayor que 1.

En consecuencia, antes de normalizar, disminuyen los pesos de los ejemplos acertados y aumentan los de los fallados.

### **5.3. Aplicación al dataset**

Partimos de:

$$
w_i=\frac1{14}\approx0.07143
$$

Y:

$$
\alpha_1\approx0.4581
$$

Para los diez días correctamente clasificados:

$$
w_{\text{acierto}}'
=
0.07143e^{-0.4581}
$$

$$
w_{\text{acierto}}'\approx0.04518
$$

Para los cuatro días mal clasificados:

$$
w_{\text{error}}'
=
0.07143e^{0.4581}
$$

$$
w_{\text{error}}'\approx0.11294
$$

Ahora calculamos la constante de normalización:

$$
Z_1=10(0.04518)+4(0.11294)
$$

$$
Z_1\approx0.90351
$$

Finalmente:

$$
w_{\text{acierto}}^{(2)}
=\frac{0.04518}{0.90351}=0.05
$$

$$
w_{\text{error}}^{(2)}
=\frac{0.11294}{0.90351}=0.125
$$

Los nuevos pesos son:

| Días | Resultado | Peso inicial | Peso nuevo |
|---|---|---:|---:|
| D1, D2 | Acierto | 0,07143 | 0,050 |
| D3, D4 | Error | 0,07143 | 0,125 |
| D5 | Acierto | 0,07143 | 0,050 |
| D6 | Error | 0,07143 | 0,125 |
| D7–D11 | Acierto | 0,07143 | 0,050 |
| D12 | Error | 0,07143 | 0,125 |
| D13, D14 | Acierto | 0,07143 | 0,050 |

Comprobamos:

$$
10(0.05)+4(0.125)=1
$$

**Interpretación:** en la segunda iteración, equivocarse con D3, D4, D6 o D12 resulta 2,5 veces más costoso que equivocarse con uno de los ejemplos que el primer clasificador acertó.

---

## **6. Segunda iteración: aprendizaje adaptativo**

AdaBoost vuelve a buscar el clasificador con menor error ponderado, pero utilizando los nuevos pesos.

Consideremos dos posibles reglas.

**Regla A**

- Si el cielo está nublado, predecir Sí.
- En caso contrario, predecir No.

Esta regla falla D4, D5, D6, D9, D10 y D11.

Su error ponderado es:

$$
\epsilon_A=2(0.125)+4(0.05)=0.45
$$

**Regla B**

- Si el viento es débil, predecir Sí.
- Si el viento es fuerte, predecir No.

Esta regla falla D1, D3, D7, D8, D11, D12 y D13.

Su error ponderado es:

$$
\epsilon_B=2(0.125)+5(0.05)=0.50
$$

Entre estas dos reglas, AdaBoost prefiere la regla A.

La razón no es solamente el número de errores, sino el peso de los ejemplos mal clasificados.

En una implementación real se evaluarían todas las reglas candidatas y se elegiría la de menor error ponderado.

Después se calcularía su coeficiente $\alpha_2$, se actualizarían nuevamente los pesos y se continuaría con la tercera iteración.

Este procedimiento se repite hasta alcanzar el número deseado de clasificadores o una condición de parada.

---

## **7. Construcción del clasificador fuerte**

AdaBoost conserva los clasificadores débiles seleccionados.

La función de decisión final es:

$$
\boxed{
F_T(x)=\sum_{t=1}^{T}\alpha_t h_t(x)
}
$$

Y la predicción:

$$
\boxed{
H(x)=\operatorname{sign}(F_T(x))
}
$$

Esto equivale a una votación ponderada.

Por ejemplo, supongamos que hemos obtenido tres clasificadores:

| Clasificador | Error ponderado | Coeficiente |
|---|---:|---:|
| C1 | 0,30 | 0,424 |
| C2 | 0,10 | 1,099 |
| C3 | 0,40 | 0,203 |

Llega un día nuevo y las predicciones son:

- C1: jugar $(+1)$.
- C2: no jugar $(-1)$.
- C3: jugar $(+1)$.

La puntuación será:

$$
F(x)=0.424(1)+1.099(-1)+0.203(1)
$$

$$
F(x)=-0.472
$$

Como la puntuación es negativa:

$$
H(x)=-1
$$

La decisión final es **no jugar al golf**.

Aunque dos clasificadores han votado jugar, el clasificador C2 tiene suficiente importancia como para determinar la decisión.

Este ejemplo de votación es ilustrativo e independiente del entrenamiento numérico anterior.

---

# **8. ¿Por qué funciona AdaBoost?**

Hasta ahora hemos explicado el procedimiento algorítmico.

Pero queda una pregunta más profunda:

**¿Por qué aumentar los pesos de los ejemplos mal clasificados y combinar clasificadores débiles puede producir un clasificador más potente?**

La respuesta se encuentra en la interpretación de AdaBoost como un procedimiento de minimización de una función de pérdida.

## **8.1. Función de decisión aditiva**

AdaBoost construye progresivamente:

$$
F_t(x)=\sum_{k=1}^{t}\alpha_k h_k(x)
$$

Podemos definir el margen de un ejemplo como:

$$
m_i=y_iF_t(x_i)
$$

El margen mide la relación entre la etiqueta real y la puntuación del clasificador.

- Si $m_i>0$, el ejemplo está correctamente clasificado.
- Si $m_i<0$, está incorrectamente clasificado.
- Cuanto mayor sea el margen positivo, mayor será la puntuación favorable a la clase correcta.

## **8.2. La pérdida exponencial**

AdaBoost puede interpretarse como un algoritmo que minimiza la función:

$$
\boxed{
L(F)=\sum_{i=1}^{N}e^{-y_iF(x_i)}
}
$$

Esta función penaliza especialmente los ejemplos con márgenes negativos.

Por ejemplo:

| Margen | Pérdida exponencial |
|---:|---:|
| −2 | 7,389 |
| −1 | 2,718 |
| 0 | 1,000 |
| +1 | 0,368 |
| +2 | 0,135 |

La pérdida aumenta rápidamente cuando el clasificador toma decisiones incorrectas con márgenes negativos elevados en valor absoluto.

Por tanto, minimizar esta pérdida implica prestar especial atención a los ejemplos difíciles.

---

## **9. Deducción matemática de la actualización de pesos**

Esta sección explica de dónde procede la regla de actualización.

### **9.1. Añadimos un nuevo clasificador**

Supongamos que ya tenemos:

$$
F_{t-1}(x)=\sum_{k=1}^{t-1}\alpha_k h_k(x)
$$

En la siguiente iteración añadimos:

$$
F_t(x)=F_{t-1}(x)+\alpha_t h_t(x)
$$

Sustituimos esta expresión en la pérdida exponencial:

$$
L(F_t)=
\sum_i e^{-y_i[F_{t-1}(x_i)+\alpha_t h_t(x_i)]}
$$

Aplicando la propiedad:

$$
e^{a+b}=e^a e^b
$$

Obtenemos:

$$
L(F_t)=
\sum_i
e^{-y_iF_{t-1}(x_i)}
e^{-\alpha_t y_i h_t(x_i)}
$$

### **9.2. Identificamos los pesos**

Definimos la contribución individual, sin normalizar, a la pérdida:

$$
q_i^{(t)}=e^{-y_iF_{t-1}(x_i)}
$$

Normalizando:

$$
\boxed{
w_i^{(t)}=
\frac{
e^{-y_iF_{t-1}(x_i)}
}{
\sum_j e^{-y_jF_{t-1}(x_j)}
}
}
$$

Esto tiene una interpretación fundamental:

> El peso de un ejemplo es proporcional a su contribución a la pérdida exponencial acumulada del clasificador combinado.

Los ejemplos con márgenes negativos elevados reciben pesos mayores.

Los ejemplos con márgenes positivos elevados reciben pesos menores.

### **9.3. Obtenemos la actualización**

Después de incorporar el nuevo clasificador:

$$
q_i^{(t+1)}=e^{-y_iF_t(x_i)}
$$

Sustituimos la expresión de $F_t$:

$$
q_i^{(t+1)}
=
e^{-y_i[F_{t-1}(x_i)+\alpha_t h_t(x_i)]}
$$

Separando los exponentes:

$$
q_i^{(t+1)}
=
e^{-y_iF_{t-1}(x_i)}
e^{-\alpha_t y_i h_t(x_i)}
$$

Por tanto:

$$
q_i^{(t+1)}
=
q_i^{(t)}e^{-\alpha_t y_i h_t(x_i)}
$$

Y, después de normalizar:

$$
\boxed{
w_i^{(t+1)}
=
\frac{
w_i^{(t)}e^{-\alpha_t y_i h_t(x_i)}
}{
Z_t
}
}
$$

**Esta es exactamente la regla de actualización de AdaBoost.**

No se trata de un factor de corrección arbitrario: aparece naturalmente al expresar cómo cambia la pérdida exponencial individual cuando añadimos un nuevo clasificador.

---

## **10. Deducción matemática del coeficiente alfa**

Ya sabemos de dónde procede la actualización de los pesos.

Ahora podemos demostrar por qué el coeficiente de cada clasificador tiene la forma:

$$
\alpha_t=\frac12\ln\left(\frac{1-\epsilon_t}{\epsilon_t}\right)
$$

Partimos de la pérdida al añadir el nuevo clasificador:

$$
L(\alpha,h)=
\sum_i w_i e^{-\alpha y_i h(x_i)}
$$

Separamos los ejemplos correctamente clasificados de los incorrectamente clasificados.

Como los pesos suman 1:

$$
L(\alpha,h)
=
(1-\epsilon)e^{-\alpha}
+
\epsilon e^\alpha
$$

Derivamos respecto a $\alpha$:

$$
\frac{\partial L}{\partial\alpha}
=
-(1-\epsilon)e^{-\alpha}
+
\epsilon e^\alpha
$$

Igualamos a cero:

$$
(1-\epsilon)e^{-\alpha}
=
\epsilon e^\alpha
$$

Reordenamos:

$$
e^{2\alpha}=\frac{1-\epsilon}{\epsilon}
$$

Aplicamos logaritmos:

$$
2\alpha=\ln\left(\frac{1-\epsilon}{\epsilon}\right)
$$

Finalmente:

$$
\boxed{
\alpha=\frac12\ln\left(\frac{1-\epsilon}{\epsilon}\right)
}
$$

Por tanto, el coeficiente alfa es el valor que minimiza la pérdida exponencial al incorporar el clasificador seleccionado.

Además, sustituyendo este valor en la función de pérdida, obtenemos:

$$
L_{\min}=2\sqrt{\epsilon(1-\epsilon)}
$$

Para errores inferiores a 0,5, esta expresión disminuye al disminuir $\epsilon$.

Esto explica por qué AdaBoost selecciona el clasificador con menor error ponderado.

---

## **11. ¿Qué garantiza matemáticamente AdaBoost?**

Supongamos que en cada iteración conseguimos un clasificador débil cuyo error ponderado es:

$$
\epsilon_t=\frac12-\gamma_t
$$

Donde $\gamma_t>0$ es su ventaja respecto a un clasificador aleatorio equilibrado.

Puede demostrarse que el error de entrenamiento del clasificador combinado está acotado por:

$$
\boxed{
\operatorname{Error}_{train}
\leq
\prod_{t=1}^{T}
2\sqrt{\epsilon_t(1-\epsilon_t)}
}
$$

Y también:

$$
\boxed{
\operatorname{Error}_{train}
\leq
\exp\left(-2\sum_{t=1}^{T}\gamma_t^2\right)
}
$$

Si todos los clasificadores mantienen una ventaja positiva suficientemente grande, la cota del error de entrenamiento disminuye exponencialmente con el número de iteraciones.

Este resultado proporciona una justificación teórica del funcionamiento de AdaBoost.

Sin embargo, no garantiza que el error sobre datos nuevos disminuya indefinidamente. El ruido, los ejemplos atípicos y el sobreajuste pueden perjudicar la generalización.

---

# **12. Implementación de AdaBoost desde cero en Python**

Vamos a implementar una versión didáctica de AdaBoost sin utilizar Scikit-learn ni ninguna biblioteca de aprendizaje automático.

Los clasificadores débiles serán reglas de la forma:

- Si un atributo tiene un valor concreto, predecir $+1$.
- En caso contrario, predecir $-1$.

También permitiremos invertir el sentido de la regla.

## **12.1. Dataset**

In [1]:
import math

datos = [
    ("Soleado", "Alta",  "Alta",   "Debil",  -1),
    ("Soleado", "Alta",  "Alta",   "Fuerte", -1),
    ("Nublado", "Alta",  "Alta",   "Debil",   1),
    ("Lluvia",  "Suave", "Alta",   "Debil",   1),
    ("Lluvia",  "Baja",  "Normal", "Debil",   1),
    ("Lluvia",  "Baja",  "Normal", "Fuerte", -1),
    ("Nublado", "Baja",  "Normal", "Fuerte",  1),
    ("Soleado", "Suave", "Alta",   "Debil",  -1),
    ("Soleado", "Baja",  "Normal", "Debil",   1),
    ("Lluvia",  "Suave", "Normal", "Debil",   1),
    ("Soleado", "Suave", "Normal", "Fuerte",  1),
    ("Nublado", "Suave", "Alta",   "Fuerte",  1),
    ("Nublado", "Alta",  "Normal", "Debil",   1),
    ("Lluvia",  "Suave", "Alta",   "Fuerte", -1)
]

atributos = ["Cielo", "Temperatura", "Humedad", "Viento"]

## **12.2. Clasificador débil**

Una regla se representa mediante tres valores:

- `columna`: atributo que queremos evaluar.
- `valor`: valor con el que comparamos.
- `polaridad`: predicción cuando coincide el valor.

In [2]:
def predecir_regla(ejemplo, columna, valor, polaridad):
    if ejemplo[columna] == valor:
        return polaridad
    return -polaridad

Por ejemplo:

In [3]:
# Si el cielo está nublado, predecir Sí
regla = (0, "Nublado", 1)

prediccion = predecir_regla(datos[2], *regla)
print(prediccion)

1


Resultado:

```text
1
```

## **12.3. Buscar el clasificador de menor error**

La siguiente función evalúa todas las reglas posibles de nuestro conjunto de clasificadores débiles.

In [4]:
def buscar_mejor_regla(datos, pesos):

    mejor_error = float("inf")
    mejor_regla = None

    for columna in range(4):

        valores = sorted(set(fila[columna] for fila in datos))

        for valor in valores:

            for polaridad in [1, -1]:

                error = 0

                for i, fila in enumerate(datos):

                    prediccion = predecir_regla(
                        fila, columna, valor, polaridad
                    )

                    if prediccion != fila[4]:
                        error += pesos[i]

                if error < mejor_error:
                    mejor_error = error
                    mejor_regla = (columna, valor, polaridad)

    return mejor_regla, mejor_error

Esta función implementa:

$$
h_t=\arg\min_h
\sum_i w_i^{(t)}
\mathbb{1}[h(x_i)\neq y_i]
$$

## **12.4. Entrenamiento de AdaBoost**

Este es el núcleo del algoritmo.

In [5]:
def adaboost(datos, iteraciones=5):

    n = len(datos)
    pesos = [1 / n] * n
    clasificadores = []

    for t in range(iteraciones):

        # 1. Seleccionar el mejor clasificador
        regla, error = buscar_mejor_regla(datos, pesos)

        if error >= 0.5:
            break

        # 2. Calcular alfa
        if error == 0:
            alpha = 1.0
        else:
            alpha = 0.5 * math.log((1 - error) / error)

        clasificadores.append((regla, alpha))

        # 3. Actualizar los pesos
        nuevos_pesos = []

        for i, fila in enumerate(datos):

            y = fila[4]
            h = predecir_regla(fila, *regla)

            nuevo_peso = pesos[i] * math.exp(-alpha * y * h)

            nuevos_pesos.append(nuevo_peso)

        # 4. Normalizar
        total = sum(nuevos_pesos)
        pesos = [w / total for w in nuevos_pesos]

        # 5. Mostrar el resultado de la iteración
        columna, valor, polaridad = regla

        print(f"\nITERACIÓN {t + 1}")
        print(f"Regla: {atributos[columna]} == {valor}")
        print(f"Polaridad: {polaridad:+d}")
        print(f"Error: {error:.4f}")
        print(f"Alfa: {alpha:.4f}")

        for i, peso in enumerate(pesos):
            print(f"D{i+1:2d}: {peso:.4f}")

        if error == 0:
            break

    return clasificadores

En esta implementación, si el error es cero, detenemos el entrenamiento y asignamos un alfa positivo finito. Es una simplificación práctica: matemáticamente, el alfa óptimo tendería a infinito.

## **12.5. Clasificación final**

Una vez entrenados los clasificadores débiles, calculamos su votación ponderada.

In [6]:
def predecir_adaboost(ejemplo, clasificadores):

    puntuacion = 0

    for regla, alpha in clasificadores:

        h = predecir_regla(ejemplo, *regla)
        puntuacion += alpha * h

    return 1 if puntuacion >= 0 else -1

## **12.6. Ejecución**

In [7]:
modelo = adaboost(datos, iteraciones=5)

aciertos = 0

for i, fila in enumerate(datos):

    prediccion = predecir_adaboost(fila, modelo)
    real = fila[4]

    if prediccion == real:
        aciertos += 1

    print(
        f"Día {i+1}: "
        f"Real={real:+d}, "
        f"Predicción={prediccion:+d}"
    )

print(f"\nAciertos: {aciertos}/{len(datos)}")


ITERACIÓN 1
Regla: Cielo == Soleado
Polaridad: -1
Error: 0.2857
Alfa: 0.4581
D 1: 0.0500
D 2: 0.0500
D 3: 0.0500
D 4: 0.0500
D 5: 0.0500
D 6: 0.1250
D 7: 0.0500
D 8: 0.0500
D 9: 0.1250
D10: 0.0500
D11: 0.1250
D12: 0.0500
D13: 0.0500
D14: 0.1250

ITERACIÓN 2
Regla: Humedad == Alta
Polaridad: -1
Error: 0.2750
Alfa: 0.4847
D 1: 0.0345
D 2: 0.0345
D 3: 0.0909
D 4: 0.0909
D 5: 0.0345
D 6: 0.2273
D 7: 0.0345
D 8: 0.0345
D 9: 0.0862
D10: 0.0345
D11: 0.0862
D12: 0.0909
D13: 0.0345
D14: 0.0862

ITERACIÓN 3
Regla: Cielo == Lluvia
Polaridad: -1
Error: 0.2633
Alfa: 0.5144
D 1: 0.0655
D 2: 0.0655
D 3: 0.0617
D 4: 0.1726
D 5: 0.0655
D 6: 0.1543
D 7: 0.0234
D 8: 0.0655
D 9: 0.0585
D10: 0.0655
D11: 0.0585
D12: 0.0617
D13: 0.0234
D14: 0.0585

ITERACIÓN 4
Regla: Viento == Debil
Polaridad: +1
Error: 0.2746
Alfa: 0.4858
D 1: 0.1192
D 2: 0.0451
D 3: 0.0425
D 4: 0.1190
D 5: 0.0451
D 6: 0.1063
D 7: 0.0426
D 8: 0.1192
D 9: 0.0403
D10: 0.0451
D11: 0.1065
D12: 0.1124
D13: 0.0161
D14: 0.0403

ITERACIÓN 5
Regla:

También podemos predecir si se jugará en un nuevo día:

In [8]:
nuevo_dia = ("Soleado", "Suave", "Normal", "Debil")

resultado = predecir_adaboost(nuevo_dia, modelo)

print("Jugar" if resultado == 1 else "No jugar")

Jugar


Esta implementación permite observar directamente los aspectos esenciales de AdaBoost: el error ponderado, la selección de reglas, los coeficientes alfa, la evolución de los pesos y la combinación final.

---

## **13. Relación con Viola-Jones**

AdaBoost fue uno de los componentes fundamentales del detector de objetos propuesto por Viola y Jones.

En ese contexto:

| AdaBoost con el dataset del golf | Viola-Jones |
|---|---|
| Ejemplo: día meteorológico | Ejemplo: ventana de imagen |
| Etiqueta: jugar/no jugar | Etiqueta: rostro/no rostro |
| Atributos meteorológicos | Características Haar |
| Regla basada en un atributo | Característica Haar, umbral y polaridad |
| Error ponderado | Error ponderado de clasificación |
| Clasificador fuerte | Combinación de clasificadores Haar |

En Viola-Jones, AdaBoost selecciona automáticamente características Haar discriminativas y combina clasificadores débiles.

Posteriormente, los clasificadores se organizan en una cascada que permite descartar rápidamente regiones que no contienen rostros.

Así, AdaBoost desempeña dos funciones importantes:

1. Seleccionar características visuales útiles.
2. Construir clasificadores fuertes a partir de reglas sencillas.

---

## **14. Resumen conceptual**

AdaBoost puede comprenderse desde tres perspectivas complementarias.

### **Perspectiva algorítmica**

AdaBoost entrena clasificadores débiles secuencialmente, aumentando la importancia relativa de los ejemplos que el último clasificador ha fallado.

### **Perspectiva estadística**

Cada clasificador se selecciona minimizando un error ponderado, y su influencia en el resultado final depende de ese error.

### **Perspectiva matemática**

AdaBoost construye una función aditiva que reduce progresivamente la pérdida exponencial:

$$
L(F)=\sum_i e^{-y_iF(x_i)}
$$

Los pesos de los ejemplos son proporcionales a su pérdida exponencial acumulada:

$$
w_i^{(t)}
\propto
e^{-y_iF_{t-1}(x_i)}
$$

Y la importancia del clasificador se obtiene minimizando esa pérdida:

$$
\alpha_t=
\frac12\ln\left(\frac{1-\epsilon_t}{\epsilon_t}\right)
$$

La actualización de los pesos:

$$
w_i^{(t+1)}
=
\frac{
w_i^{(t)}e^{-\alpha_t y_i h_t(x_i)}
}{
Z_t
}
$$

es, por tanto, una consecuencia natural de la forma exponencial de la función objetivo.

**La idea más importante es que AdaBoost no se limita a corregir los errores del último clasificador: construye progresivamente un modelo combinado y modifica los pesos según los márgenes acumulados de sus predicciones.**

